##  Fine-Tuning DistilBERT for POS Tagging & Chunking
## NLP Assignment 5
- Task: Token Classification — POS Tagging & Chunking
- Dataset: CoNLL-2003
- Model: DistilBERT (distilbert-base-uncased)
- Tools: Python · HuggingFace Transformers · PyTorch · seqevalase-level grouping labels

#  Task 1: Dataset Selection

## Dataset Chosen: CoNLL-2003 (Sample Version)

For this assignment, a CoNLL-2003 formatted dataset is used for performing:

- Part-of-Speech (POS) Tagging
- Chunking (Phrase Detection)

# Dataset Creation

In [5]:
# CoNLL-style dataset (lightweight for demonstration)

train_data = [
    [("EU", "NNP", "B-NP"), ("rejects", "VBZ", "B-VP"), ("German", "JJ", "B-NP"), ("call", "NN", "I-NP")],
    [("Peter", "NNP", "B-NP"), ("works", "VBZ", "B-VP"), ("at", "IN", "B-PP"), ("Google", "NNP", "B-NP")],
    [("The", "DT", "B-NP"), ("quick", "JJ", "I-NP"), ("fox", "NN", "I-NP"), ("jumps", "VBZ", "B-VP")]
]

val_data = train_data
test_data = train_data

print("Dataset Loaded Successfully")
print("Number of sentences:", len(train_data))
print("Sample sentence:", train_data[0])

Dataset Loaded Successfully
Number of sentences: 3
Sample sentence: [('EU', 'NNP', 'B-NP'), ('rejects', 'VBZ', 'B-VP'), ('German', 'JJ', 'B-NP'), ('call', 'NN', 'I-NP')]


In [8]:
# Extract unique POS and Chunk labels

pos_labels = sorted(list(set(pos for sent in train_data for _, pos, _ in sent)))
chunk_labels = sorted(list(set(chunk for sent in train_data for _, _, chunk in sent)))

print("Labels Extracted")
print("POS Labels:", pos_labels)
print("Chunk Labels:", chunk_labels)

Labels Extracted
POS Labels: ['DT', 'IN', 'JJ', 'NN', 'NNP', 'VBZ']
Chunk Labels: ['B-NP', 'B-PP', 'B-VP', 'I-NP']


In [10]:
import pandas as pd

sample = train_data[0]

df = pd.DataFrame(sample, columns=["Token", "POS Tag", "Chunk Tag"])

print("Sample Sentence with Labels:")
print(df)

Sample Sentence with Labels:
     Token POS Tag Chunk Tag
0       EU     NNP      B-NP
1  rejects     VBZ      B-VP
2   German      JJ      B-NP
3     call      NN      I-NP


#  Task 2: Data Preprocessing

In this step, we prepare the dataset for transformer models.

In [13]:
!pip install transformers -q

In [16]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print("Tokenizer Loaded")

Tokenizer Loaded


In [18]:
# Converting the dataset into tokens + labels format

def prepare_data(data):
    tokens = []
    pos_tags = []
    chunk_tags = []

    for sent in data:
        tokens.append([w[0] for w in sent])
        pos_tags.append([w[1] for w in sent])
        chunk_tags.append([w[2] for w in sent])

    return tokens, pos_tags, chunk_tags

tokens, pos_tags, chunk_tags = prepare_data(train_data)

print("Sample Tokens:", tokens[0])
print("Sample POS:", pos_tags[0])

Sample Tokens: ['EU', 'rejects', 'German', 'call']
Sample POS: ['NNP', 'VBZ', 'JJ', 'NN']


In [20]:
# Create label mappings

pos2id = {label: i for i, label in enumerate(pos_labels)}
chunk2id = {label: i for i, label in enumerate(chunk_labels)}

id2pos = {i: label for label, i in pos2id.items()}
id2chunk = {i: label for label, i in chunk2id.items()}

print("POS mapping:", pos2id)

POS mapping: {'DT': 0, 'IN': 1, 'JJ': 2, 'NN': 3, 'NNP': 4, 'VBZ': 5}


In [22]:
def tokenize_and_align(tokens, labels, label2id):
    tokenized = tokenizer(
        tokens,
        is_split_into_words=True,
        truncation=True
    )

    word_ids = tokenized.word_ids()

    aligned_labels = []
    previous_word_idx = None

    for word_idx in word_ids:
        if word_idx is None:
            aligned_labels.append(-100)
        elif word_idx != previous_word_idx:
            aligned_labels.append(label2id[labels[word_idx]])
        else:
            aligned_labels.append(-100)

        previous_word_idx = word_idx

    return tokenized, aligned_labels

In [24]:
sample_tokens = tokens[0]
sample_labels = pos_tags[0]

tokenized, aligned = tokenize_and_align(sample_tokens, sample_labels, pos2id)

print("Tokens after tokenization:")
print(tokenizer.convert_ids_to_tokens(tokenized["input_ids"]))

print("\nAligned Labels:")
print(aligned)

Tokens after tokenization:
['[CLS]', 'eu', 'rejects', 'german', 'call', '[SEP]']

Aligned Labels:
[-100, 4, 5, 2, 3, -100]


# TASK 3- MODEL SETUP

In [29]:
from transformers import AutoModelForTokenClassification

In [31]:
pos_model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(pos_labels),
    id2label=id2pos,
    label2id=pos2id
)

print("POS Model Loaded")

W0405 20:49:13.082000 33552 anaconda3\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


POS Model Loaded


In [35]:
chunk_model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(chunk_labels),
    id2label=id2chunk,
    label2id=chunk2id
)

print("Chunk Model Loaded")

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Chunk Model Loaded


In [37]:
print("POS Labels:", pos_model.config.num_labels)
print("Chunk Labels:", chunk_model.config.num_labels)

POS Labels: 6
Chunk Labels: 4


# Task 4: Model Training

In this step, we train the BERT model for token classification.

## Key Parameters:
- Learning Rate: 2e-5
- Epochs: 1 (kept low for fast execution)
- Batch Size: 2 (small dataset)

In [40]:
import torch

def prepare_training_data(tokens_list, labels_list, label2id):
    encodings = []
    all_labels = []

    for tokens, labels in zip(tokens_list, labels_list):
        tokenized, aligned_labels = tokenize_and_align(tokens, labels, label2id)

        encodings.append(tokenized)
        all_labels.append(aligned_labels)

    return encodings, all_labels

In [42]:
pos_encodings, pos_labels_aligned = prepare_training_data(tokens, pos_tags, pos2id)

print("POS Data Ready")

POS Data Ready


In [44]:
class SimpleDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val) for key, val in self.encodings[idx].items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

pos_dataset = SimpleDataset(pos_encodings, pos_labels_aligned)

print("Dataset size:", len(pos_dataset))

Dataset size: 3


In [48]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    num_train_epochs=1,
    logging_steps=1,
    report_to="none"
)

print("TrainingArguments Ready")

TrainingArguments Ready


In [50]:
from transformers import Trainer

trainer_pos = Trainer(
    model=pos_model,
    args=training_args,
    train_dataset=pos_dataset
)

print("Trainer Ready")

Trainer Ready


In [52]:
trainer_pos.train()

print("Training Completed")

C:\Users\karth\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
1,1.621400
2,1.727300


Training Completed


# Task-5 Model Evaluation 

In [56]:
!pip install seqeval -q

In [60]:
# Install 
!pip install seqeval -q

# Imports
import numpy as np
from seqeval.metrics import classification_report, f1_score

# Get predictions
predictions, labels, _ = trainer_pos.predict(pos_dataset)

# Convert logits → label ids
preds = np.argmax(predictions, axis=2)

# Convert ids → actual labels
true_labels = []
true_preds = []

for pred, label in zip(preds, labels):
    temp_true = []
    temp_pred = []

    for p, l in zip(pred, label):
        if l != -100:  # ignore special tokens
            temp_true.append(id2pos[l])
            temp_pred.append(id2pos[p])

    true_labels.append(temp_true)
    true_preds.append(temp_pred)

# Print results
print("Evaluation Results\n")

print("F1 Score:", f1_score(true_labels, true_preds))

print("\nClassification Report:")
print(classification_report(true_labels, true_preds))

C:\Users\karth\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Evaluation Results

F1 Score: 0.6086956521739131

Classification Report:
              precision    recall  f1-score   support

          BZ       0.50      0.33      0.40         3
           J       0.50      1.00      0.67         2
           N       1.00      0.33      0.50         3
          NP       1.00      0.67      0.80         3
           T       0.50      1.00      0.67         1

   micro avg       0.64      0.58      0.61        12
   macro avg       0.70      0.67      0.61        12
weighted avg       0.75      0.58      0.59        12



C:\Users\karth\anaconda3\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: NNP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
C:\Users\karth\anaconda3\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: VBZ seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
C:\Users\karth\anaconda3\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: JJ seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
C:\Users\karth\anaconda3\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: NN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
C:\Users\karth\anaconda3\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: IN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
C:\Users\karth\anaconda3\Lib\site-packages\seqeval\metrics\sequence_label

#  Task 6: Inference
The model predicts:
- POS Tags (word-level grammar)
- Chunk Tags (phrase-level grouping)

In [63]:
import torch

def predict(sentence, model, tokenizer, id2label):
    words = sentence.split()

    encoding = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True
    )

    model.eval()
    with torch.no_grad():
        outputs = model(**encoding)

    preds = torch.argmax(outputs.logits, dim=2)[0].numpy()
    word_ids = encoding.word_ids()

    result = []
    used = set()

    for i, word_id in enumerate(word_ids):
        if word_id is None or word_id in used:
            continue
        result.append((words[word_id], id2label[preds[i]]))
        used.add(word_id)

    return result


# Test sentence
sentence = "John works at Google"

print("Input Sentence:", sentence)

# POS Prediction
pos_result = predict(sentence, pos_model, tokenizer, id2pos)

print("\nPOS Tagging:")
for word, tag in pos_result:
    print(f"{word} → {tag}")

Input Sentence: John works at Google

POS Tagging:
John → JJ
works → JJ
at → IN
Google → NNP


# Task 7: Comparison — POS Tagging vs Chunking

## POS Tagging
- Assigns grammatical role to each word
- Examples: Noun (NN), Verb (VB), Adjective (JJ)
- Works at word-level
- Easier task

## Chunking
- Groups words into meaningful phrases
- Examples: Noun Phrase (NP), Verb Phrase (VP)
- Works at phrase-level
- Slightly more complex

## Key Difference

| Feature        | POS Tagging        | Chunking            |
|---------------|------------------|--------------------|
| Level         | Word-level        | Phrase-level       |
| Complexity    | Easy              | Medium             |
| Output        | Grammar tags      | Phrase groups      |

## Conclusion
POS tagging identifies grammatical roles, while chunking identifies structure of sentences.

# Task 8: Report

## Differences between POS Tagging and Chunking

POS tagging focuses on identifying the grammatical role of each word in a sentence. 
Chunking, on the other hand, groups words into phrases such as noun phrases and verb phrases.

## Challenges Faced

- Handling tokenization and subwords
- Aligning labels with tokens correctly
- Managing special tokens like [CLS] and [SEP]
- Dealing with environment and dependency issues

## Observations

- Token classification is sensitive to preprocessing
- Label alignment is the most important step
- Even small datasets can demonstrate model behavior
- Transformer models like BERT are powerful for NLP tasks

## Conclusion

This project demonstrates how transformer models can be used for sequence labeling tasks like POS tagging and chunking. Proper preprocessing and label alignment are key to achieving good performance.